# Exploração e Qualidade de Dados — Tabelas NCM
**Projeto:** analise-importacoes-brasil  
**Fonte:** Receita Federal do Brasil — Nomenclatura Comum do Mercosul  
**Portal:** https://www.gov.br/receitafederal/pt-br/assuntos/aduana-e-comercio-exterior/classificacao-fiscal-de-mercadorias/download-ncm-nomenclatura-comum-do-mercosul

Este notebook:
1. Lê e normaliza os três arquivos baixados da RFB
2. Recomenda e justifica o arquivo para uso no projeto
3. Executa checagens de qualidade de dados
4. Gera `dados_ref/correlacao_ncm.csv` para uso no pipeline

---
## 0. Dependências

In [1]:
# pip install pandas openpyxl

import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', 20)

print('Dependências carregadas.')

Dependências carregadas.


In [2]:
# Ajuste os caminhos conforme a pasta onde você salvou os arquivos
PASTA_ARQUIVOS = Path('K:/Download/dados_importacoes_temp')   # <-- ajuste se necessário
PASTA_DADOS_REF = Path('../dados_ref')

ARQ_VIGENTE    = PASTA_ARQUIVOS / 'Tabela_NCM_Vigente_20261119.xlsx'
ARQ_CONCAT     = PASTA_ARQUIVOS / 'Tabela_NCM_Desc_Concatenada_Vigente_20261119.xlsx'
ARQ_HISTORICO  = PASTA_ARQUIVOS / 'Tabela_NCM_Desc_Concatenada_Historica_20261119.xlsx'

# Data de corte da série de importações
DATA_CORTE = pd.Timestamp('2022-04-01')

print(f'Pasta dos arquivos : {PASTA_ARQUIVOS}')
print(f'Dados de referência: {PASTA_DADOS_REF}')
print(f'Corte da série     : {DATA_CORTE.date()}')

Pasta dos arquivos : K:\Download\dados_importacoes_temp
Dados de referência: ..\dados_ref
Corte da série     : 2022-04-01


---
## 1. Leitura e normalização dos três arquivos

Os arquivos da RFB têm 3 linhas de cabeçalho antes dos dados. A quarta linha contém os nomes reais das colunas.

In [3]:
def ler_tabela_ncm(caminho: Path, renomear_duplicadas: bool = False) -> pd.DataFrame:
    """
    Lê um arquivo Excel de NCM da RFB, pula as 3 linhas de cabeçalho
    e usa a 4ª linha como nomes de coluna.
    """
    df = pd.read_excel(caminho, skiprows=3, dtype=str)
    df.columns = df.iloc[0].tolist()
    df = df.iloc[1:].reset_index(drop=True)
    df = df.dropna(how='all')

    if renomear_duplicadas:
        # O histórico tem duas colunas 'Número' e 'Ano' (início e fim)
        cols = df.columns.tolist()
        cols[8]  = 'Ato Legal Fim'
        cols[9]  = 'Numero Fim'
        cols[10] = 'Ano Fim'
        df.columns = cols

    # Normaliza nomes de colunas
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace('ó', 'o', regex=False)
        .str.replace('í', 'i', regex=False)
        .str.replace('ç', 'c', regex=False)
        .str.replace('ã', 'a', regex=False)
    )

    # Converte datas
    for col in ['data_inicio', 'data_fim']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')

    return df


df_vigente   = ler_tabela_ncm(ARQ_VIGENTE)
df_concat    = ler_tabela_ncm(ARQ_CONCAT)
df_historico = ler_tabela_ncm(ARQ_HISTORICO, renomear_duplicadas=True)

print('Arquivos lidos com sucesso.')
print(f'  Vigente   : {df_vigente.shape}')
print(f'  Concatenado: {df_concat.shape}')
print(f'  Histórico : {df_historico.shape}')

Arquivos lidos com sucesso.
  Vigente   : (15157, 7)
  Concatenado: (15157, 8)
  Histórico : (269, 11)


---
## 2. Comparação dos três arquivos

In [4]:
print('=== Vigente ===')
print(f'Colunas : {df_vigente.columns.tolist()}')
print(f'Linhas  : {len(df_vigente):,}')
display(df_vigente.head(3))

print('\n=== Descrição Concatenada (Vigente) ===')
print(f'Colunas : {df_concat.columns.tolist()}')
print(f'Linhas  : {len(df_concat):,}')
display(df_concat.head(3))

print('\n=== Histórico ===')
print(f'Colunas : {df_historico.columns.tolist()}')
print(f'Linhas  : {len(df_historico):,}')
display(df_historico.head(3))

=== Vigente ===
Colunas : ['codigo', 'descricao', 'data_inicio', 'data_fim', 'ato_legal_inicio', 'número', 'ano']
Linhas  : 15,157


,codigo,descricao,data_inicio,data_fim,ato_legal_inicio,número,ano
0,01,Animais vivos.,2022-04-01,9999-12-31,Res Gecex,272,2021
1,01.01,"Cavalos, asininos e muares, vivos.",2022-04-01,9999-12-31,Res Gecex,272,2021
2,0101.2,- Cavalos:,2022-04-01,9999-12-31,Res Gecex,272,2021



=== Descrição Concatenada (Vigente) ===
Colunas : ['codigo', 'descricao', 'descricao_concatenada', 'data_inicio', 'data_fim', 'ato_legal_inicio', 'número', 'ano']
Linhas  : 15,157


,codigo,descricao,descricao_concatenada,data_inicio,data_fim,ato_legal_inicio,número,ano
0,01,Animais vivos.,01 Animais vivos.,2022-04-01,9999-12-31,Res Gecex,272,2021
1,01.01,"Cavalos, asininos e muares, vivos.","01 Animais vivos. \n01.01 Cavalos, asininos e muares, vivos.",2022-04-01,9999-12-31,Res Gecex,272,2021
2,0101.2,- Cavalos:,"01 Animais vivos. \n01.01 Cavalos, asininos e muares, vivos. \n0101.2 - Cava...",2022-04-01,9999-12-31,Res Gecex,272,2021



=== Histórico ===
Colunas : ['codigo', 'descricao', 'descricao_concatenada', 'data_inicio', 'data_fim', 'ato_legal_inicio', 'número', 'ano', 'ato_legal_fim', 'numero_fim', 'ano_fim']
Linhas  : 269


,codigo,descricao,descricao_concatenada,data_inicio,data_fim,ato_legal_inicio,número,ano,ato_legal_fim,numero_fim,ano_fim
0,0207.12,"-- Não cortadas em pedaços, congeladas","0207.12 -- Não cortadas em pedaços, congeladas",2023-04-01,9999-12-31,Res Gecex,440,2022,NaN,NaN,NaN
1,0207.12.00,"-- Não cortadas em pedaços, congeladas","0207.12 -- Não cortadas em pedaços, congeladas \n0207.12.00 -- Não cortadas ...",2022-04-01,2023-03-31,Res Gecex,272,2021,Res Gecex,440,2022
2,0207.12.10,Com miudezas,"0207.12 -- Não cortadas em pedaços, congeladas \n0207.12.10 Com miudezas",2023-04-01,9999-12-31,Res Gecex,440,2022,NaN,NaN,NaN


In [5]:
# Resumo comparativo dos três arquivos
resumo = pd.DataFrame({
    'Arquivo': [
        'Tabela_NCM_Vigente',
        'Tabela_NCM_Desc_Concatenada_Vigente',
        'Tabela_NCM_Desc_Concatenada_Historica',
    ],
    'Linhas': [
        len(df_vigente),
        len(df_concat),
        len(df_historico),
    ],
    'Colunas': [
        len(df_vigente.columns),
        len(df_concat.columns),
        len(df_historico.columns),
    ],
    'Tem histórico de mudanças': ['Não', 'Não', 'Sim'],
    'Tem descrição concatenada': ['Não', 'Sim', 'Sim'],
    'Recomendado para o projeto': ['Complementar', 'Complementar', '✅ Principal'],
})
display(resumo)

,Arquivo,Linhas,Colunas,Tem histórico de mudanças,Tem descrição concatenada,Recomendado para o projeto
0,Tabela_NCM_Vigente,15157,7,Não,Não,Complementar
1,Tabela_NCM_Desc_Concatenada_Vigente,15157,8,Não,Sim,Complementar
2,Tabela_NCM_Desc_Concatenada_Historica,269,11,Sim,Sim,✅ Principal


### Por que o arquivo histórico é o mais indicado

O projeto analisa a série de importações desde abril/2022. Nesse período, a RFB alterou vários NCMs, dividindo, fundindo ou renumerando códigos. Sem mapear essas mudanças, um produto pode parecer sumir de um código e aparecer em outro, simulando um efeito que não existe.

O arquivo histórico registra exatamente isso:
- Quais NCMs vigeram e por qual período (`data_inicio` → `data_fim`)
- Qual ato legal encerrou cada código (`ato_legal_fim`)

O arquivo vigente entra como complemento para identificar o código substituto de cada NCM encerrado.

---
## 3. Exploração do arquivo histórico

In [6]:
# Tipos de código por número de dígitos (sem pontos)
df_historico['codigo_limpo'] = df_historico['codigo'].str.replace('.', '', regex=False)
df_historico['n_digitos']    = df_historico['codigo_limpo'].str.len()

print('Distribuição por nível hierárquico (nº de dígitos sem pontos):')
contagem = df_historico['n_digitos'].value_counts().sort_index().reset_index()
contagem.columns = ['Dígitos', 'Quantidade']
contagem['Nível'] = contagem['Dígitos'].map({
    2: 'Capítulo',
    4: 'Posição',
    6: 'Subposição',
    8: 'NCM (folha)',
})
display(contagem)

Distribuição por nível hierárquico (nº de dígitos sem pontos):


,Dígitos,Quantidade,Nível
0,6,21,Subposição
1,7,24,NaN
2,8,224,NCM (folha)


In [7]:
# Datas de início presentes na série
print('Datas de início dos NCMs no histórico:')
print(sorted(df_historico['data_inicio'].dropna().dt.strftime('%d/%m/%Y').unique()))

print('\nDatas de fim:')
print(sorted(df_historico['data_fim'].dropna().dt.strftime('%d/%m/%Y').unique()))

Datas de início dos NCMs no histórico:
['01/01/2023', '01/01/2024', '01/01/2025', '01/02/2026', '01/04/2022', '01/04/2023', '01/04/2024', '01/07/2022', '01/07/2024', '01/08/2024', '01/09/2022', '01/10/2024', '01/10/2025', '01/10/2026', '01/11/2023']

Datas de fim:
['30/06/2022', '30/06/2024', '30/09/2024', '30/09/2026', '31/01/2026', '31/03/2023', '31/03/2024', '31/07/2024', '31/08/2022', '31/10/2023', '31/12/2022', '31/12/2023', '31/12/2024', '31/12/9999']


In [8]:
# NCMs encerrados (data_fim != 31/12/9999) vs ainda vigentes
DATA_FIM_INDEFINIDA = pd.Timestamp('9999-12-31')

encerrados = df_historico[df_historico['data_fim'] < DATA_FIM_INDEFINIDA]
vigentes   = df_historico[df_historico['data_fim'] == DATA_FIM_INDEFINIDA]

print(f'NCMs encerrados no período  : {len(encerrados)}')
print(f'NCMs ainda vigentes         : {len(vigentes)}')

print('\nNível dos NCMs encerrados (nº de dígitos):')
print(encerrados['n_digitos'].value_counts().sort_index().to_string())

print('\nExemplos de NCMs encerrados:')
display(encerrados[['codigo','descricao','data_inicio','data_fim','ato_legal_fim']].head(8))

NCMs encerrados no período  : 74
NCMs ainda vigentes         : 195

Nível dos NCMs encerrados (nº de dígitos):
n_digitos
7     3
8    71

Exemplos de NCMs encerrados:


,codigo,descricao,data_inicio,data_fim,ato_legal_fim
1,0207.12.00,"-- Não cortadas em pedaços, congeladas",2022-04-01,2023-03-31,Res Gecex
5,0207.14.00,"-- Pedaços e miudezas, congelados",2022-04-01,2024-07-31,Res Gecex
24,0302.91.00,"-- Fígados, ovas e gônadas masculinas",2022-04-01,2023-03-31,Res Gecex
28,0303.91.00,"-- Fígados, ovas e gônadas masculinas",2022-04-01,2023-03-31,Res Gecex
32,0305.20.00,"- Fígados, ovas e gônadas masculinas, de peixes, secos, defumados (fumados),...",2022-04-01,2023-03-31,Res Gecex
35,0712.90.10,Alho em pó,2022-04-01,2024-12-31,Res Gecex
38,1513.21.10,De amêndoa de palma (palmiste) (coconote),2022-04-01,2022-06-30,Res Gecex
42,1513.29.10,De amêndoa de palma (palmiste) (coconote),2022-04-01,2022-06-30,Res Gecex


In [9]:
# Foco em NCMs nível folha (8 dígitos) — os que aparecem nos CSVs de importação
ncm_folha_hist = df_historico[df_historico['n_digitos'] == 8].copy()
ncm_folha_enc  = ncm_folha_hist[ncm_folha_hist['data_fim'] < DATA_FIM_INDEFINIDA]

print(f'NCMs folha (8 dígitos) no histórico : {len(ncm_folha_hist)}')
print(f'  Encerrados                         : {len(ncm_folha_enc)}')
print(f'  Ainda vigentes                     : {len(ncm_folha_hist) - len(ncm_folha_enc)}')

print('\nNCMs folha encerrados no período:')
display(ncm_folha_enc[['codigo','descricao','data_inicio','data_fim']].to_string())

NCMs folha (8 dígitos) no histórico : 224
  Encerrados                         : 71
  Ainda vigentes                     : 153

NCMs folha encerrados no período:


'         codigo                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

---
## 4. Checagens de qualidade de dados

In [10]:
erros  = []
avisos = []

print('=' * 55)
print('CHECAGENS DE QUALIDADE — ARQUIVO HISTÓRICO')
print('=' * 55)

# 1. Nulos em colunas obrigatórias
obrigatorias = ['codigo', 'descricao', 'data_inicio', 'data_fim']
for col in obrigatorias:
    nulos = df_historico[col].isna().sum()
    if nulos > 0:
        erros.append(f'Nulos em "{col}": {nulos} linha(s)')
    else:
        print(f'✅ "{col}": sem nulos')

# 2. Código vazio
cod_vazio = (df_historico['codigo'].str.strip() == '').sum()
if cod_vazio > 0:
    erros.append(f'Códigos vazios: {cod_vazio}')
else:
    print('✅ Nenhum código vazio')

# 3. Data início posterior à data fim
inv = (df_historico['data_inicio'] > df_historico['data_fim']).sum()
if inv > 0:
    erros.append(f'data_inicio > data_fim: {inv} linha(s)')
else:
    print('✅ data_inicio <= data_fim: todas as linhas válidas')

# 4. Registros anteriores ao corte da série
antes_corte = (df_historico['data_inicio'] < DATA_CORTE).sum()
if antes_corte > 0:
    avisos.append(
        f'{antes_corte} registro(s) com data_inicio anterior a {DATA_CORTE.date()} '
        '— podem ser ignorados na correlacao_ncm.csv'
    )
else:
    print(f'✅ Todos os registros têm data_inicio >= {DATA_CORTE.date()}')

# 5. Duplicatas de código + período
dup = df_historico.duplicated(subset=['codigo','data_inicio','data_fim']).sum()
if dup > 0:
    erros.append(f'Chave (codigo + data_inicio + data_fim) duplicada: {dup} linha(s)')
else:
    print('✅ Chave (codigo + data_inicio + data_fim): sem duplicatas')

# 6. Código limpo com tamanho inesperado
tamanhos_esperados = {2, 4, 6, 8}
tamanhos_inesperados = df_historico[~df_historico['n_digitos'].isin(tamanhos_esperados)]
if len(tamanhos_inesperados) > 0:
    avisos.append(
        f'{len(tamanhos_inesperados)} código(s) com tamanho fora do padrão '
        f'(2/4/6/8 dígitos): {tamanhos_inesperados["codigo"].tolist()}'
    )
else:
    print('✅ Todos os códigos têm 2, 4, 6 ou 8 dígitos (padrão NCM)')

# 7. NCMs folha (8 dígitos) com data_fim = 9999 e ato_legal_fim preenchido
# (inconsistência: encerrado pelo ato mas data_fim indica vigência indefinida)
incons = df_historico[
    (df_historico['n_digitos'] == 8) &
    (df_historico['data_fim'] == DATA_FIM_INDEFINIDA) &
    (df_historico['ato_legal_fim'].notna())
]
if len(incons) > 0:
    avisos.append(
        f'{len(incons)} NCM(s) folha com data_fim=9999 mas ato_legal_fim preenchido'
    )
    display(incons[['codigo','descricao','data_fim','ato_legal_fim']])
else:
    print('✅ Sem inconsistências entre data_fim e ato_legal_fim')

# Relatório
print('\n' + '=' * 55)
if erros:
    print(f'\n❌ ERROS ({len(erros)}):')
    for e in erros: print(f'   • {e}')
else:
    print('\n✅ Nenhum erro encontrado.')

if avisos:
    print(f'\n⚠️  AVISOS ({len(avisos)}):')
    for a in avisos: print(f'   • {a}')
else:
    print('✅ Nenhum aviso.')

CHECAGENS DE QUALIDADE — ARQUIVO HISTÓRICO
✅ "codigo": sem nulos
✅ "descricao": sem nulos
✅ "data_inicio": sem nulos
✅ "data_fim": sem nulos
✅ Nenhum código vazio
✅ data_inicio <= data_fim: todas as linhas válidas
✅ Todos os registros têm data_inicio >= 2022-04-01
✅ Chave (codigo + data_inicio + data_fim): sem duplicatas
✅ Sem inconsistências entre data_fim e ato_legal_fim


✅ Nenhum erro encontrado.

⚠️  AVISOS (1):
   • 24 código(s) com tamanho fora do padrão (2/4/6/8 dígitos): ['0207.14.1', '0207.14.2', '0207.14.3', '1513.21.1', '1513.29.1', '2827.39.3', '2929.90.2', '2929.90.3', '2931.49.3', '3302.90.9', '3824.99.6', '3906.90.4', '3906.90.5', '3906.90.6', '3907.29.9', '4002.20.9', '4811.90.1', '7409.40.1', '8443.32.9', '8443.32.9', '8505.90.1', '8544.19.1', '8714.93.1', '8714.96.1']


---
## 5. Cruzamento histórico × vigente

Para cada NCM encerrado no histórico, identifica se existe um código substituto na tabela vigente.

In [12]:
# NCMs folha vigentes (da tabela vigente, 8 dígitos)
df_vigente['codigo_limpo'] = df_vigente['codigo'].str.replace('.', '', regex=False)
ncms_vigentes = set(df_vigente[df_vigente['codigo_limpo'].str.len() == 8]['codigo_limpo'])

print(f'NCMs folha vigentes hoje: {len(ncms_vigentes):,}')

# NCMs encerrados no histórico que NÃO existem mais na tabela vigente
encerrados_folha = ncm_folha_enc.copy()
encerrados_folha['existe_no_vigente'] = encerrados_folha['codigo_limpo'].isin(ncms_vigentes)

print(f'NCMs encerrados que ainda existem no vigente (renumeração parcial): '
      f'{encerrados_folha["existe_no_vigente"].sum()}')
print(f'NCMs encerrados que não existem mais no vigente (extintos/fundidos): '
      f'{(~encerrados_folha["existe_no_vigente"]).sum()}')

print('\nNCMs encerrados e extintos/fundidos:')
display(encerrados_folha[~encerrados_folha['existe_no_vigente']][
    ['codigo','descricao','data_inicio','data_fim']
])

NCMs folha vigentes hoje: 10,516
NCMs encerrados que ainda existem no vigente (renumeração parcial): 19
NCMs encerrados que não existem mais no vigente (extintos/fundidos): 52

NCMs encerrados e extintos/fundidos:


,codigo,descricao,data_inicio,data_fim
1,0207.12.00,"-- Não cortadas em pedaços, congeladas",2022-04-01,2023-03-31
5,0207.14.00,"-- Pedaços e miudezas, congelados",2022-04-01,2024-07-31
24,0302.91.00,"-- Fígados, ovas e gônadas masculinas",2022-04-01,2023-03-31
28,0303.91.00,"-- Fígados, ovas e gônadas masculinas",2022-04-01,2023-03-31
32,0305.20.00,"- Fígados, ovas e gônadas masculinas, de peixes, secos, defumados (fumados),...",2022-04-01,2023-03-31
35,0712.90.10,Alho em pó,2022-04-01,2024-12-31
38,1513.21.10,De amêndoa de palma (palmiste) (coconote),2022-04-01,2022-06-30
42,1513.29.10,De amêndoa de palma (palmiste) (coconote),2022-04-01,2022-06-30
46,1901.20.00,"- Misturas e pastas para a preparação de produtos de padaria, pastelaria e d...",2022-04-01,2023-12-31
55,2710.91.00,"-- Que contenham bifenilas policloradas (PCB), terfenilas policloradas (PCT)...",2022-04-01,2024-03-31
